---

# 🔍 Visão Geral do Pipeline

## Fluxo Completo de Processamento

```
┌─────────────────────────────────────────────────────────────────────────┐
│                         PIPELINE COMPLETO                                │
├─────────────────────────────────────────────────────────────────────────┤
│  1. COLETA DE DADOS                                                      │
│     API NLR/NSRDB → GHI diário (2019-2024)                               │
│                          │                                               │
│                          ▼                                               │
│  2. PRÉ-PROCESSAMENTO                                                    │
│     Limpeza → Quantização (128 níveis) → Normalização [0,1]             │
│                          │                                               │
│                          ▼                                               │
│  3. FEATURES TEMPORAIS                                                   │
│     Lags (1,2,3,7) + Médias Móveis (3,7,30)                              │
│                          │                                               │
│                          ▼                                               │
│  4. DIVISÃO TREINO/TESTE                                                 │
│     80% treino │ 20% teste (cronológico)                                 │
│                          │                                               │
│              ┌───────────┴───────────┐                                   │
│              ▼                       ▼                                   │
│  5. TREINAMENTO              6. AVALIAÇÃO                                │
│     XGBoost                    MAE, MSE, RMSE, R²                        │
│     MLP                                                                  │
└─────────────────────────────────────────────────────────────────────────┘
```

## As 6 Etapas do Pipeline

| Etapa | Descrição | Arquivo |
|-------|-----------|---------|
| **1. Coleta** | Obtenção de dados da API NLR/NSRDB | `preprocessamento.py` |
| **2. Pré-processamento** | Limpeza, quantização, normalização | `preprocessamento.py` |
| **3. Features** | Criação de lags e médias móveis | `features.py` |
| **4. Divisão** | Separação treino/teste cronológica | `features.py` |
| **5. Modelos** | Treinamento XGBoost e MLP | `modelos.py` |
| **6. Avaliação** | Cálculo de métricas e gráficos | `avaliacao.py`, `graficos.py` |

---

# 1️⃣ Coleta de Dados: API NLR/NSRDB (2019-2024)

## 📌 O que é esta etapa?

A primeira etapa do pipeline é responsável por **obter os dados brutos** de irradiância solar que serão usados para treinar os modelos.

## 🏛️ API NLR/NSRDB

**NLR** = *National Laboratory of the Rockies* (denominação atual do antigo NREL)

**NSRDB** = *National Solar Radiation Database*

### Características da API:

| Característica | Descrição |
|----------------|-----------|
| **Fonte** | Dados de satélite e estações terrestres |
| **Cobertura** | Global (com maior precisão nas Américas) |
| **Resolução Temporal** | Horária (30 min, 60 min) |
| **Resolução Espacial** | 2km x 2km a 10km x 10km |
| **Período disponível no produto utilizado** | 1998 - 2024 no momento da coleta |

## 📅 Período 2019-2024

- ✅ **6 anos de dados**: Suficiente para capturar padrões sazonais
- ✅ **Dados recentes**: Condições climáticas atuais
- ✅ **Consistência**: Mesma tecnologia de medição
- ✅ **Completo**: Sem grandes lacunas de dados

## 🌍 Localidades do Projeto

| # | Localidade | País | Latitude | Longitude |
|---|------------|------|----------|-----------|
| 1 | BYD Camacari | Brasil | -12.67° | -38.28° |
| 2 | Tesla Gigafactory Nevada | EUA | 39.54° | -119.44° |
| 3 | Tesla Gigafactory Texas | EUA | 30.22° | -97.62° |
| 4 | Hyundai Metaplant Georgia | EUA | 32.16° | -81.45° |
| 5 | Rivian Normal | EUA | 40.51° | -89.05° |
| 6 | Tesla Fremont Factory | EUA | 37.49° | -121.94° |
| 7 | Lucid AMP 1 Casa Grande | EUA | 32.86° | -111.78° |
| 8 | GM Factory Zero | EUA | 42.38° | -83.05° |
| 9 | Ford Rouge EV Center | EUA | 42.31° | -83.17° |
| 10 | BMW San Luis Potosi | México | 21.97° | -100.85° |

---

# 2️⃣ Pré-processamento: Limpeza, Quantização e Normalização

## 🧹 2.1 Limpeza dos Dados

### Problemas Comuns em Dados Brutos:

| Problema | Descrição | Solução |
|----------|-----------|----------|
| **Valores NaN** | Dados faltantes | Remover linhas |
| **Valores Negativos** | GHI não pode ser negativo | Remover (filtro ≥ 0) |
| **Duplicatas** | Mesma data repetida | Manter última ocorrência |
| **Fora de Ordem** | Dados não cronológicos | Ordenar por data |

### Exemplo Prático:

**Antes da Limpeza:**

| data | ghi |
|------|-----|
| 2023-01-01 | 200 |
| 2023-01-02 | NaN |
| 2023-01-02 | 180 |
| 2023-01-03 | -10 |
| 2023-01-04 | 220 |

**Depois da Limpeza:**

| data | ghi |
|------|-----|
| 2023-01-01 | 200 |
| 2023-01-02 | 180 |
| 2023-01-04 | 220 |

> ⚠️ NaN foi removido, duplicata consolidada, valor negativo removido.

---

## 📊 2.2 Quantização em 128 Níveis

### O que é Quantização?

**Quantização** é o processo de converter valores **contínuos** em valores **discretos**.

```
Contínuo:  0, 1, 2, ... 398, 399, 400 (infinitos valores possíveis)
              ↓↓ Quantização ↓↓
Discreto:  0, 1, 2, ... 126, 127 (apenas 128 valores)
```

### Por que Quantizar?

| Vantagem | Explicação |
|----------|------------|
| **Redução de Ruído** | Pequenas variações são agrupadas |
| **Generalização** | Modelo aprende padrões, não valores exatos |
| **Robustez** | Menos sensível a outliers |

### Fórmula da Quantização:

```python
nivel = round((valor - minimo) / (maximo - minimo) * 127)
```

### Exemplo Numérico:

Suponha: mínimo = 0, máximo = 400, níveis = 128

| GHI Original | Cálculo | Nível |
|--------------|---------|-------|
| 0 W/m² | (0-0)/(400-0) * 127 = 0 | **0** |
| 200 W/m² | (200-0)/(400-0) * 127 = 63.5 | **64** |
| 400 W/m² | (400-0)/(400-0) * 127 = 127 | **127** |
| 100 W/m² | (100-0)/(400-0) * 127 = 31.75 | **32** |

---

## 📐 2.3 Normalização Min-Max

### O que é Normalização Min-Max?

**Normalização Min-Max** escala os dados para um intervalo específico, geralmente **[0, 1]**.

### Por que Normalizar?

| Razão | Explicação |
|-------|------------|
| **Redes Neurais** | MLP funciona melhor com dados em [0, 1] |
| **Convergência** | Gradientes mais estáveis durante treino |
| **Comparabilidade** | Todas as features na mesma escala |

### Fórmula da Normalização:

```python
valor_normalizado = (valor - minimo) / (maximo - minimo)
```

### Exemplo Numérico:

Após quantização, valores vão de 0 a 127:

| Quantizado | Cálculo | Normalizado |
|------------|---------|-------------|
| 0 | (0-0)/(127-0) = 0/127 | **0.000** |
| 32 | (32-0)/(127-0) = 32/127 | **0.252** |
| 64 | (64-0)/(127-0) = 64/127 | **0.504** |
| 95 | (95-0)/(127-0) = 95/127 | **0.748** |
| 127 | (127-0)/(127-0) = 127/127 | **1.000** |

### ⚠️ Importante: Sem Vazamento de Dados!

Os parâmetros (mínimo, máximo) são calculados **apenas no treino**:

```python
# CORRETO:
minimo_treino = treino['ghi'].min()
teste_quantizado = quantizar(teste['ghi'], minimo=minimo_treino)

# ERRADO (vazamento):
minimo_geral = todos_dados['ghi'].min()  # ❌ Inclui teste!
```

---

# 3️⃣ Features Temporais: Lags e Médias Móveis

## 📊 3.1 Lags (Defasagens)

### O que são Lags?

**Lag** = valor da série em um momento **anterior** no tempo.

### Lags Usados neste Projeto:

| Feature | Significado | Propósito |
|---------|-------------|-----------|
| `ghi_t-1` | GHI de ontem | Tendência imediata |
| `ghi_t-2` | GHI de 2 dias atrás | Curto prazo |
| `ghi_t-3` | GHI de 3 dias atrás | Curto prazo |
| `ghi_t-7` | GHI de 1 semana atrás | Padrão semanal |

### Por que estes valores?

- **1, 2, 3 dias**: Capturam a tendência recente
- **7 dias**: Captura padrão semanal

---

## 📈 3.2 Médias Móveis

### O que são Médias Móveis?

**Média Móvel** = média dos valores em uma janela de tempo deslizante.

### Médias Móveis Usadas:

| Feature | Janela | Propósito |
|---------|--------|-----------|
| `ghi_media_movel_3d` | 3 dias | Tendência de curto prazo |
| `ghi_media_movel_7d` | 7 dias | Tendência semanal |
| `ghi_media_movel_30d` | 30 dias | Tendência mensal |

### ⚠️ Importante: Alinhamento com o Alvo

```python
# CORRETO: a linha de data t preve t+1
# A janela termina em t e usa somente dias anteriores ao alvo.
media_3d = dados['ghi_normalizado'].rolling(window=3).mean()

# ERRADO: deslocar para o futuro incluiria o valor-alvo na entrada.
media_3d = dados['ghi_normalizado'].shift(-1).rolling(window=3).mean()  # ❌
```

## 🎯 Resumo da Etapa 3

- **4 lags** + **3 médias móveis** = **7 features temporais**

---

# 4️⃣ Divisão Treino/Teste: 80%/20% Cronológico

## 📅 Por que Divisão CRONOLÓGICA?

### ⚠️ Em Séries Temporais, NÃO se Embaralha!

```
┌─────────────────────────────────────────────────────────────┐
│                    DIVISÃO CRONOLÓGICA                       │
├─────────────────────────────────────────────────────────────┤
│  Dia 1 ──────────────► Dia 800 │ Dia 801 ─────► Dia 1000   │
│         TREINO (80%)          │        TESTE (20%)          │
│                                                              │
│         O modelo aprende com o PASSADO                       │
│         e é avaliado no FUTURO                               │
└─────────────────────────────────────────────────────────────┘
```

### Por que não embaralhar?

| Problema | Explicação |
|----------|------------|
| **Vazamento Temporal** | Modelo veria dados do "futuro" no treino |
| **Avaliação Irreal** | Métricas seriam otimistas demais |
| **Não reflete uso real** | Na prática, prevemos o futuro, não o passado |

## 🎯 Resumo da Etapa 4

| Item | Descrição |
|------|-----------|
| **Proporção** | 80% treino, 20% teste |
| **Método** | Divisão cronológica (sem shuffle) |
| **Propósito** | Avaliar capacidade de generalização |

---

# 5️⃣ Modelos: XGBoost e MLPRegressor

## 🌳 5.1 XGBoost

**XGBoost** = *Extreme Gradient Boosting*

É um algoritmo de **ensemble** que combina múltiplas **árvores de decisão**.

### Hiperparâmetros Usados:

```python
XGBRegressor(
    n_estimators=300,      # 300 árvores
    max_depth=3,           # Árvores rasas (3 níveis)
    learning_rate=0.05,    # Aprendizado conservador
    subsample=0.9,         # 90% dos dados por árvore
    colsample_bytree=0.9,  # 90% das features por árvore
    objective='reg:squarederror',
    random_state=42,
    n_jobs=-1
)
```

### Vantagens do XGBoost:

| Vantagem | Explicação |
|----------|------------|
| **Alta Performance** | Um dos melhores para dados tabulares |
| **Robusto** | Lida bem com outliers e ruídos |
| **Rápido** | Treinamento eficiente |
| **Interpretável** | Importância das features disponível |

---

## 🧠 5.2 MLPRegressor (Rede Neural)

**MLP** = *Multi-Layer Perceptron*

É uma **rede neural artificial** com múltiplas camadas de neurônios.

### Arquitetura Usada:

```
Entrada (7 features)
    │
    ▼
┌─────────────────┐
│ Camada Oculta 1 │  64 neurônios, ativação ReLU
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│ Camada Oculta 2 │  32 neurônios, ativação ReLU
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│    Saída (1)    │  Previsão de GHI
└─────────────────┘
```

### Hiperparâmetros Usados:

```python
MLPRegressor(
    hidden_layer_sizes=(64, 32),  # 2 camadas: 64 e 32 neurônios
    activation='relu',            # Função ReLU
    solver='adam',                # Otimizador Adam
    max_iter=1000,                # Máximo de iterações
    learning_rate_init=0.001,     # Taxa de aprendizado
    random_state=42
)
```

## 📊 Comparação XGBoost vs MLP

| Característica | XGBoost | MLP |
|----------------|---------|-----|
| **Tipo** | Ensemble de árvores | Rede neural |
| **Treinamento** | Mais rápido | Mais lento |
| **Interpretabilidade** | Alta | Baixa (caixa preta) |
| **Dados Tabulares** | Excelente | Bom |
| **Padrões Não-lineares** | Bom | Excelente |

---

# 6️⃣ Métricas: MAE, MSE, RMSE, R²

## 📊 6.1 MAE - Erro Absoluto Médio

### Fórmula:

```
MAE = (1/n) × Σ |y_real - y_previsto|
```

### Interpretação:

| Valor MAE | Interpretação |
|-----------|---------------|
| 0.05 | Erro médio de 5% da escala |
| 0.10 | Erro médio de 10% da escala |
| 0.20 | Erro médio de 20% da escala |

---

## 📊 6.2 MSE - Erro Quadrático Médio

### Fórmula:

```
MSE = (1/n) × Σ (y_real - y_previsto)²
```

### Característica:
- **Penaliza erros grandes**: Erro quadrático cresce rápido

---

## 📊 6.3 RMSE - Raiz do Erro Quadrático Médio

### Fórmula:

```
RMSE = √MSE
```

### Vantagem:
- ✅ **Mesma escala** da variável avaliada; neste projeto, a escala normalizada [0, 1]

---

## 📊 6.4 R² - Coeficiente de Determinação

### Fórmula:

```
R² = 1 - (SSE / SST)

Onde:
- SSE = Σ(y_real - y_previsto)²  (Soma dos Erros)
- SST = Σ(y_real - y_media)²     (Variância total)
```

### Interpretação:

| Valor R² | Interpretação |
|----------|---------------|
| 1.0 | Perfeito (prevê exatamente) |
| 0.8-0.9 | Muito bom |
| 0.5-0.7 | Moderado |
| 0.0 | Igual a prever a média |
| negativo | Pior que a média |

## 📋 Resumo das Métricas

| Métrica | Fórmula | Intervalo | Melhor Valor |
|---------|---------|-----------|--------------|
| **MAE** | média(\|erro\|) | [0, ∞) | 0 |
| **MSE** | média(erro²) | [0, ∞) | 0 |
| **RMSE** | √MSE | [0, ∞) | 0 |
| **R²** | 1 - SSE/SST | (-∞, 1] | 1 |

---

# 🎓 Conclusão

## Resumo do Pipeline Completo

```
┌─────────────────────────────────────────────────────────────────────┐
│                        PIPELINE COMPLETO                             │
├─────────────────────────────────────────────────────────────────────┤
│  1. COLETA                                                           │
│     API NLR → Dados brutos de GHI (2019-2024)                       │
│                          │                                           │
│                          ▼                                           │
│  2. PRÉ-PROCESSAMENTO                                                │
│     Limpeza → Quantização (128 níveis) → Normalização [0,1]         │
│                          │                                           │
│                          ▼                                           │
│  3. FEATURES                                                         │
│     Lags (1,2,3,7) + Médias Móveis (3,7,30)                         │
│                          │                                           │
│                          ▼                                           │
│  4. TREINO/TESTE                                                     │
│     80% treino │ 20% teste (cronológico)                            │
│                          │                                           │
│              ┌─────────────┴─────────────┐                          │
│              ▼                           ▼                          │
│  5. MODELOS                    6. AVALIAÇÃO                          │
│     XGBoost (300 árvores)       MAE, MSE, RMSE, R²                  │
│     MLP (64-32 neurônios)       Gráficos comparativos               │
└─────────────────────────────────────────────────────────────────────┘
```

## Principais Conceitos Aprendidos

| Conceito | Importância |
|----------|-------------|
| **Quantização** | Reduz ruído, melhora generalização |
| **Normalização** | Essencial para redes neurais |
| **Lags** | Captura dependência temporal |
| **Médias Móveis** | Suaviza variações, mostra tendência |
| **Divisão Cronológica** | Evita vazamento temporal |
| **XGBoost** | Excelente para dados tabulares |
| **MLP** | Captura padrões não-lineares complexos |
| **Métricas** | Avaliação completa do desempenho |

---

## 📚 Referências

- **XGBoost**: https://xgboost.readthedocs.io/
- **Scikit-learn MLP**: https://scikit-learn.org/stable/modules/neural_networks_supervised.html
- **PVLib**: https://pvlib-python.readthedocs.io/
- **NLR NSRDB**: https://developer.nlr.gov/docs/solar/nsrdb/

---

<div align="center">

**Material criado para apresentação do TCC**

Previsão e Geração de Séries Temporais de GHI usando XGBoost e MLP

</div>